# CMS reference data exploration

This notebook inspects the version-pinned CMS sources selected for the project as of November 23, 2026:

- FY 2027 ICD-10-CM code descriptions, effective October 1, 2026 through September 30, 2027.
- October 2026 Alpha-Numeric HCPCS Level II file, effective October 1 through December 31, 2026.
- RVU26D Medicare Physician Fee Schedule October release, effective for the selected Q4 2026 snapshot.

Sources: [CMS ICD-10 files](https://www.cms.gov/medicare/coding-billing/ICD-10-codes), [CMS HCPCS quarterly updates](https://www.cms.gov/medicare/coding-billing/healthcare-common-procedure-system/quarterly-update), and [CMS RVU26D](https://www.cms.gov/medicare/payment/fee-schedules/physician/pfs-relative-value-files/rvu26d).

The notebook reads directly from the preserved ZIP archives. It does not modify or extract the raw source files.

In [1]:
from __future__ import annotations

import hashlib
import io
import json
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)

def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/raw/cms/_inventory.json").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from within the project repository.")

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

REPOSITORY_ROOT = find_repository_root()
CMS_DIRECTORY = REPOSITORY_ROOT / "data/raw/cms"
inventory = json.loads((CMS_DIRECTORY / "_inventory.json").read_text(encoding="utf-8"))
source_by_id = {item["dataset_id"]: item for item in inventory["datasets"]}
print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Inventory validated at: {inventory['validated_at_utc']}")

Repository root: C:\Users\krishna\OneDrive\Desktop\chandhra lekha\healthcare-revenue-cycle-reconciliation
Inventory validated at: 2026-11-23T14:10:53.985094+00:00


## Source inventory and archive integrity

In [2]:
inventory_columns = [
    "dataset_id", "title", "filename", "release_date",
    "effective_start", "effective_end", "size_bytes", "member_count", "sha256",
]
inventory_frame = pd.DataFrame(inventory["datasets"])[inventory_columns]
display(inventory_frame)

integrity_checks = []
for item in inventory["datasets"]:
    archive_path = CMS_DIRECTORY / item["filename"]
    with zipfile.ZipFile(archive_path) as archive:
        corrupt_member = archive.testzip()
    integrity_checks.append(
        {
            "dataset_id": item["dataset_id"],
            "exists": archive_path.is_file(),
            "size_matches": archive_path.stat().st_size == item["size_bytes"],
            "sha256_matches": sha256(archive_path) == item["sha256"],
            "zip_crc_valid": corrupt_member is None,
        }
    )
integrity_frame = pd.DataFrame(integrity_checks)
display(integrity_frame)
assert integrity_frame.drop(columns="dataset_id").all(axis=None)

,dataset_id,title,filename,release_date,effective_start,effective_end,size_bytes,member_count,sha256
0,icd10cm_fy2027,FY 2027 ICD-10-CM Code Descriptions in Tabular Order,2027-code-descriptions-tabular-order.zip,not stated on CMS source page,2026-10-01,2027-09-30,2208022,6,91c6c9d1117764ce72375a0f3a5493b1725dafbc1ab283b55799076c9e194965
1,hcpcs_2026_q4,October 2026 Alpha-Numeric HCPCS File,october-2026-alpha-numeric-hcpcs-file.zip,2026-09-23,2026-10-01,2026-12-31,2518755,7,08c2202fa92c1d1fb6f31b808b61c767bc487954f6930b6a0f8d09c89af5992f
2,mpfs_rvu26d,2026 Medicare Physician Fee Schedule RVU26D October Release,rvu26d-updated-08-26-2026.zip,2026-08-26,2026-10-01,2026-12-31,6181360,19,521ff0f4ecbf13b5d99dc1dc04d9f82e5361e55c6b4b6b3033043598bf8406e6


,dataset_id,exists,size_matches,sha256_matches,zip_crc_valid
0,icd10cm_fy2027,True,True,True,True
1,hcpcs_2026_q4,True,True,True,True
2,mpfs_rvu26d,True,True,True,True


## ICD-10-CM diagnosis codes

In [3]:
icd_source = source_by_id["icd10cm_fy2027"]
icd_path = CMS_DIRECTORY / icd_source["filename"]
icd_member = "Code Descriptions/icd10cm_codes_2027.txt"
with zipfile.ZipFile(icd_path) as archive:
    icd_lines = archive.read(icd_member).decode("latin-1").splitlines()

icd_codes = pd.DataFrame(
    {
        "code": [line[:7].strip() for line in icd_lines],
        "description": [line[8:].strip() for line in icd_lines],
    }
)
icd_summary = pd.DataFrame(
    {
        "metric": ["rows", "columns", "blank codes", "duplicate codes", "codes containing decimal"],
        "value": [
            len(icd_codes),
            ", ".join(icd_codes.columns),
            int(icd_codes["code"].eq("").sum()),
            int(icd_codes["code"].duplicated().sum()),
            int(icd_codes["code"].str.contains(r"\\.", regex=True).sum()),
        ],
    }
)
display(icd_summary)
display(icd_codes.head())
print("Effective dates are release-level metadata, not columns in the code-description file.")

,metric,value
0,rows,74879
1,columns,"code, description"
2,blank codes,0
3,duplicate codes,0
4,codes containing decimal,0


,code,description
0,A000,"Cholera due to Vibrio cholerae 01, biovar cholerae"
1,A001,"Cholera due to Vibrio cholerae 01, biovar eltor"
2,A009,"Cholera, unspecified"
3,A0100,"Typhoid fever, unspecified"
4,A0101,Typhoid meningitis


Effective dates are release-level metadata, not columns in the code-description file.


## HCPCS Level II procedure and supply codes

In [4]:
hcpcs_source = source_by_id["hcpcs_2026_q4"]
hcpcs_path = CMS_DIRECTORY / hcpcs_source["filename"]
hcpcs_member = "HCPC2026_OCT_ANWEB_09232026.xlsx"
with zipfile.ZipFile(hcpcs_path) as archive:
    hcpcs_workbook = io.BytesIO(archive.read(hcpcs_member))
hcpcs = pd.read_excel(hcpcs_workbook, dtype=str, engine="openpyxl")

hcpcs_date_columns = ["ADD DT", "ACT EFF DT", "TERM DT"]
hcpcs_dates = hcpcs[hcpcs_date_columns].apply(
    lambda column: pd.to_datetime(column, format="%Y%m%d", errors="coerce")
)
hcpcs_summary = pd.DataFrame(
    {
        "metric": ["rows", "columns", "code field", "duplicate codes", "date fields"],
        "value": [
            len(hcpcs),
            len(hcpcs.columns),
            "HCPC",
            int(hcpcs["HCPC"].duplicated().sum()),
            ", ".join(hcpcs_date_columns),
        ],
    }
)
display(hcpcs_summary)
display(pd.DataFrame({"column": hcpcs.columns}))
display(hcpcs[["HCPC", "LONG DESCRIPTION", "SHORT DESCRIPTION", *hcpcs_date_columns]].head())
display(hcpcs_dates.agg(["min", "max", "count"]))

,metric,value
0,rows,9154
1,columns,48
2,code field,HCPC
3,duplicate codes,0
4,date fields,"ADD DT, ACT EFF DT, TERM DT"


,column
0,HCPC
1,SEQNUM
2,RECID
3,LONG DESCRIPTION
4,SHORT DESCRIPTION
5,PRICE1
6,PRICE2
7,PRICE3
8,PRICE4
9,MULT_PI


,HCPC,LONG DESCRIPTION,SHORT DESCRIPTION,ADD DT,ACT EFF DT,TERM DT
0,A1,Dressing for one wound,Dressing for one wound,20020701,20020701,NaN
1,A2,Dressing for two wounds,Dressing for two wounds,20020701,20020701,NaN
2,A3,Dressing for three wounds,Dressing for three wounds,20020701,20020701,NaN
3,A4,Dressing for four wounds,Dressing for four wounds,20020701,20020701,NaN
4,A5,Dressing for five wounds,Dressing for five wounds,20020701,20020701,NaN


,ADD DT,ACT EFF DT,TERM DT
min,1982-01-01 00:00:00,1984-01-01 00:00:00,2012-03-31 00:00:00
max,2026-10-01 00:00:00,2026-10-01 00:00:00,2026-09-30 00:00:00
count,9154,9037,1342


## Medicare Physician Fee Schedule RVU26D

In [5]:
mpfs_source = source_by_id["mpfs_rvu26d"]
mpfs_path = CMS_DIRECTORY / mpfs_source["filename"]
mpfs_member = "PPRRVU2026_Oct_nonQPP.csv"
with zipfile.ZipFile(mpfs_path) as archive:
    mpfs_bytes = archive.read(mpfs_member)

mpfs_header_rows = pd.read_csv(
    io.BytesIO(mpfs_bytes), header=None, nrows=10, dtype=str, keep_default_na=False
)
mpfs = pd.read_csv(io.BytesIO(mpfs_bytes), skiprows=9, dtype=str)
display(mpfs_header_rows)
display(pd.DataFrame({"parsed_column": mpfs.columns}))
display(mpfs.head())

mpfs_code_type = mpfs["HCPCS"].fillna("").str.fullmatch(r"\\d{5}").map(
    {True: "five-digit CPT-style", False: "other HCPCS format"}
)
mpfs_summary = pd.DataFrame(
    {
        "metric": ["rows", "parsed columns", "code field", "duplicate code/modifier pairs", "effective-date columns"],
        "value": [
            len(mpfs),
            len(mpfs.columns),
            "HCPCS",
            int(mpfs.duplicated(subset=["HCPCS", "MOD"]).sum()),
            "none",
        ],
    }
)
display(mpfs_summary)
display(mpfs_code_type.value_counts().rename_axis("code representation").to_frame("rows"))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
0,,,2026 National Physician Fee Schedule Relative Value File October Release,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,,,CPT codes and descriptions only are copyright 2026 American Medical Associat...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,,,Dental codes (D codes) are copyright 2026 American Dental Association. All R...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,,,RELEASED 08/26/2026,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
5,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,NON-FACILITY,FACILITY,
6,,,,,NOT USED,,,,,,,,,,,,,,,,,,,,,,PHYSICIAN,,DIAGNOSTIC,PE USED,PE USED,MP USED
7,,,,,FOR,,,NON-FAC,,FACILITY,,,,,,,,,,,,,,,,,SUPERVISION OF,,IMAGING,FOR OPPS,FOR OPPS,FOR OPPS
8,,,,STATUS,MEDICARE,WORK,NON-FAC,NA,FACILITY,NA,MP,NON-FACILITY,FACILITY,PCTC,GLOB,PRE,INTRA,POST,MULT,BILAT,ASST,CO-,TEAM,PRIC,ENDO,CONV,DIAGNOSTIC,CALCULATION,FAMILY,PAYMENT,PAYMENT,PAYMENT
9,HCPCS,MOD,DESCRIPTION,CODE,PAYMENT,RVU,PE RVU,INDICATOR,PE RVU,INDICATOR,RVU,TOTAL,TOTAL,IND,DAYS,OP,OP,OP,PROC,SURG,SURG,SURG,SURG,IND,BASE,FACTOR,PROCEDURES,FLAG,INDICATOR,AMOUNT,AMOUNT,AMOUNT


,parsed_column
0,HCPCS
1,MOD
2,DESCRIPTION
3,CODE
4,PAYMENT
5,RVU
6,PE RVU
7,INDICATOR
8,PE RVU.1
9,INDICATOR.1


,HCPCS,MOD,DESCRIPTION,CODE,PAYMENT,RVU,PE RVU,INDICATOR,PE RVU.1,INDICATOR.1,RVU.1,TOTAL,TOTAL.1,IND,DAYS,OP,OP.1,OP.2,PROC,SURG,SURG.1,SURG.2,SURG.3,IND.1,BASE,FACTOR,PROCEDURES,FLAG,INDICATOR.2,AMOUNT,AMOUNT.1,AMOUNT.2
0,0001F,NaN,Heart failure composite,I,NaN,0.00,0.00,NaN,0.00,NaN,0.00,0.00,0.00,9,XXX,0.00,0.00,0.00,9,9,9,9,9,9,NaN,33.4009,09,0,99,0.00,0.00,0.00
1,0005F,NaN,Osteoarthritis composite,I,NaN,0.00,0.00,NaN,0.00,NaN,0.00,0.00,0.00,9,XXX,0.00,0.00,0.00,9,9,9,9,9,9,NaN,33.4009,09,0,99,0.00,0.00,0.00
2,00100,NaN,Anes px salivary gland w/bx,J,NaN,0.00,0.00,NaN,0.00,NaN,0.00,0.00,0.00,9,XXX,0.00,0.00,0.00,9,9,9,9,9,9,NaN,33.4009,09,0,99,0.00,0.00,0.00
3,00102,NaN,Anes px plstc rpr cleft lip,J,NaN,0.00,0.00,NaN,0.00,NaN,0.00,0.00,0.00,9,XXX,0.00,0.00,0.00,9,9,9,9,9,9,NaN,33.4009,09,0,99,0.00,0.00,0.00
4,00103,NaN,Anes rcnstv px eyelid,J,NaN,0.00,0.00,NaN,0.00,NaN,0.00,0.00,0.00,9,XXX,0.00,0.00,0.00,9,9,9,9,9,9,NaN,33.4009,09,0,99,0.00,0.00,0.00


,metric,value
0,rows,19453
1,parsed columns,32
2,code field,HCPCS
3,duplicate code/modifier pairs,0
4,effective-date columns,none


,rows
code representation,
other HCPCS format,19453


## Format and field comparison

In [6]:
format_comparison = pd.DataFrame(
    [
        {
            "dataset": "FY 2027 ICD-10-CM",
            "primary format": "headerless fixed-width text in ZIP",
            "code field": "positions 1-7; decimal omitted",
            "description field": "position 9 onward",
            "effective-date fields": "none at row level",
            "version rule": "fiscal year",
        },
        {
            "dataset": "October 2026 HCPCS Level II",
            "primary format": "Excel and fixed-width text in ZIP",
            "code field": "HCPC",
            "description field": "LONG DESCRIPTION / SHORT DESCRIPTION",
            "effective-date fields": "ADD DT, ACT EFF DT, TERM DT",
            "version rule": "quarterly",
        },
        {
            "dataset": "RVU26D MPFS",
            "primary format": "CSV, text, and Excel in ZIP",
            "code field": "HCPCS plus optional MOD",
            "description field": "DESCRIPTION",
            "effective-date fields": "none at row level",
            "version rule": "calendar-year quarterly release",
        },
    ]
)
display(format_comparison)

,dataset,primary format,code field,description field,effective-date fields,version rule
0,FY 2027 ICD-10-CM,headerless fixed-width text in ZIP,positions 1-7; decimal omitted,position 9 onward,none at row level,fiscal year
1,October 2026 HCPCS Level II,Excel and fixed-width text in ZIP,HCPC,LONG DESCRIPTION / SHORT DESCRIPTION,"ADD DT, ACT EFF DT, TERM DT",quarterly
2,RVU26D MPFS,"CSV, text, and Excel in ZIP",HCPCS plus optional MOD,DESCRIPTION,none at row level,calendar-year quarterly release


## Findings and limitations

- The datasets do not share a schema, delimiter, header convention, code-field name, or effective-date representation.
- ICD-10-CM code descriptions are fixed-width and omit display decimals. Validity comes from the release-level FY 2027 effective window, not a row-level date.
- The HCPCS workbook provides code-level add, active-effective, and termination dates. Blank termination dates generally indicate no termination date in this release; they are not converted to an invented future date.
- The MPFS primary CSV has a multi-row presentation header. A simple one-row parse produces repeated names such as `PE RVU.1`; transformations must map fields using the CMS layout/documentation rather than assuming those generated names are stable.
- MPFS rows may contain five-digit CPT Level I codes and HCPCS Level II codes. CPT code descriptions are copyrighted by the American Medical Association. This project does not claim to provide a comprehensive licensed CPT reference set.
- MPFS RVUs and payment indicators are Medicare reference values. They are not payer contracts, negotiated rates, allowed amounts, or proof of underpayment.
- QPP and non-QPP MPFS files are separate in RVU26D. This exploration uses the complete non-QPP file and retains the QPP file in the original archive for later, explicitly scoped work.